In [28]:
from dinosaw import PEModel
from dinosaw.utils import load_image, normalize, resize_crop, do_2D_pca, convert_image, probe, closest_crop
from dinosaw.models.vit_wrapper import (
    PretrainedViTWrapper,
    MODEL_LIST,
)
from dinosaw.datasets.translate_featurise import translate_featurise
import matplotlib.pyplot as plt
import cv2
import torch

In [29]:
model = PEModel.load_from_checkpoint("/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/dinosaw/trained_models_in_steps/batch_8_added_alibi_after_training_on_unchanged-epoch=958-val_loss=0.02_last_epoch.ckpt", strict=False).half()
model = PEModel.load_from_checkpoint("/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/dinosaw/trained_models_in_steps/batch_8_two_phase_drop_out_1_0.05_0-epoch=02-val_loss=0.04.ckpt", strict=False).half()
vit_wrapper = PretrainedViTWrapper(MODEL_LIST[1], add_flash_attn=True, device="cuda").half()

/home/ab_aimd_anja_20884/anaconda3/envs/flash_attn/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['vit.model.blocks.0.attn.m', 'vit.model.blocks.1.attn.m', 'vit.model.blocks.2.attn.m', 'vit.model.blocks.3.attn.m', 'vit.model.blocks.4.attn.m', 'vit.model.blocks.5.attn.m', 'vit.model.blocks.6.attn.m', 'vit.model.blocks.7.attn.m', 'vit.model.blocks.8.attn.m', 'vit.model.blocks.9.attn.m', 'vit.model.blocks.10.attn.m', 'vit.model.blocks.11.attn.m']


# Different Shape images

In [30]:
diff_img_size=224 # 224, 518, 1036
diff_shapes = load_image(f"images/diff_shapes_{diff_img_size}.png", resize_crop((diff_img_size, diff_img_size), (diff_img_size, diff_img_size)))[0]

diff_shapes_dino, diff_shapes_test= (
    vit_wrapper.forward_features(diff_shapes, make_2D=True),
    model(diff_shapes),
)

In [31]:
cleaned = translate_featurise(diff_shapes, vit_wrapper)

In [32]:
%%capture
fig, ax = plt.subplots(1,3, figsize=(15, 5),dpi=100)
ax[0].imshow(do_2D_pca(diff_shapes_dino.squeeze(), post_norm="minmax"))
ax[0].set_title("DINOv2")
ax[1].imshow(do_2D_pca(diff_shapes_test.squeeze(), post_norm="minmax"))
ax[1].set_title("model in testing")
ax[2].imshow(do_2D_pca(cleaned.squeeze(), post_norm="minmax"))
ax[2].set_title("DINO cleaned through translation")

In [33]:
zero_tensor = torch.zeros_like(diff_shapes)
our_black = model(zero_tensor)
dino_black = vit_wrapper.forward_features(zero_tensor, make_2D=True)

In [34]:
%%capture
fig, ax = plt.subplots(1,2)
ax[0].imshow(do_2D_pca(our_black.squeeze(), post_norm="minmax"))
ax[1].imshow(do_2D_pca(dino_black.squeeze(), post_norm="minmax"))

In [35]:
%%capture
for rmp in ["radial", "diag", "lr", "ud"]:
    probe([dino_black, our_black], [False]*2, ["DINOv2", "test"], ramp=rmp, by_channel=False, mask_step=3)

# default image (focusing on colour?)

In [36]:
img_size=518*4
test_img = load_image("images/default_image.jpg", resize_crop((img_size, img_size), (img_size, img_size)))[0]
res_dino, res_test= (
    vit_wrapper.forward_features(test_img, make_2D=True),
    model(test_img),
)

In [37]:
cleaned_res = None
#cleaned_res = translate_featurise(test_img, vit_wrapper, max_batch_size=64)

In [38]:
%%capture
if cleaned_res is not None:
    fig, ax = plt.subplots(1,4, figsize=(15, 5),dpi=300)
    ax[0].imshow(normalize(test_img.cpu().squeeze().permute(1,2,0).float()))
    ax[0].set_title("Input")
    ax[1].imshow(do_2D_pca(res_dino.squeeze(), post_norm="minmax"))
    ax[1].set_title("DINOv2")
    ax[2].imshow(do_2D_pca(res_test.squeeze(), post_norm="minmax"))
    ax[2].set_title("model in testing")
    ax[3].imshow(do_2D_pca(cleaned_res.squeeze(), post_norm="minmax"))
    ax[3].set_title("DINO cleaned through translation")
else:
    fig, ax = plt.subplots(1,3, figsize=(15, 5),dpi=300)
    ax[0].imshow(normalize(test_img.cpu().squeeze().permute(1,2,0).float()))
    ax[0].set_title("Input")
    ax[1].imshow(do_2D_pca(res_dino.squeeze(), post_norm="minmax"))
    ax[1].set_title("DINOv2")
    ax[2].imshow(do_2D_pca(res_test.squeeze(), post_norm="minmax"))
    ax[2].set_title("model in testing")

In [39]:
%%capture
for rmp in ["radial", "diag", "lr", "ud"]:
    probe([res_dino, res_test], [False]*2, ["DINOv2", "test"], ramp=rmp, by_channel=False, mask_step=3)